<h1><a name="0">Getting Started With Python for Quant Finance</a></h1>
<h2>Module #4: Treat Your Backtest Like an Experiment</h2>
<p>This code performs backtesting and walk-forward analysis using the VectorBT library for financial data. It downloads stock price data, calculates technical indicators, and identifies entry and exit points for trading strategies. The code then runs a backtest on historical data to evaluate performance. It also performs a walk-forward analysis to test the robustness of the trading strategy on unseen data, optimizing parameters like moving average windows. Finally, it assesses the statistical significance of the results using t-tests.</p>
<strong>Jupyter notebooks environment</strong>
<ul>
   <li>Jupyter notebooks allow creating and sharing documents that contain both code and rich text cells. If you are not familiar with Jupyter notebooks, read more <a href="https://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html"></a></li>
   <li>Run each code cell to see its output <strong>from top to bottom</strong>. To run a cell, click within the cell and press <strong>Shift/Command+Enter</strong>, or click <strong>Run</strong> from the top of the page menu.</li>
   <li>A <span style="font-family:monospace;">[*]</span> symbol next to the cell indicates the code is still running. A <span style="font-family:monospace;">[#]</span> symbol, where # is an integer, indicates it is finished.</li>
   <li>Beware, <strong>some code cells might take longer to run</strong>, depending on the task, installing packages and libraries, training models, etc.</li>
</ul>
<p>Please work top to bottom of this notebook and don't skip sections as this could lead to error messages due to missing code.</p>

In [ ]:
import warnings
import numpy as np
import pandas as pd
import scipy.stats as stats
import vectorbt as vbt
warnings.filterwarnings("ignore")

Define the start and end dates for downloading stock price data

In [ ]:
start = "2015"
end = "now"

Download stock price data for specified tickers between start and end dates

In [ ]:
prices = vbt.YFData.download(
    ["META", "AAPL", "AMZN", "NFLX", "GOOG"], 
    start=start,
    end=end
)

Extract the closing prices from the downloaded data

In [ ]:
prices = prices.get("Close")
prices.dropna(inplace=True)

In [ ]:
prices

Calculate 10-day and 30-day moving averages using VectorBT's built-in technical indicators

In [ ]:
fast_ma = vbt.MA.run(prices, 10, short_name="fast")
slow_ma = vbt.MA.run(prices, 30, short_name="slow")

Identify entry points where the fast moving average crosses above the slow moving average

In [ ]:
entries = fast_ma.ma_crossed_above(slow_ma)
entries

Identify exit points where the fast moving average crosses below the slow moving average

In [ ]:
exits = fast_ma.ma_crossed_below(slow_ma)
exits

Run the backtest using the identified entry and exit points and the price data

In [ ]:
pf = vbt.Portfolio.from_signals(prices, entries, exits, freq="1d")

Display the performance statistics of the backtest

In [ ]:
pf.stats()

## Time to optimize
Becuase VectorBT can simulate millions of runs in seconds, it's perfectly suited for walk forward analysis. Walk forward analysis (also called cross-fold validation), is a technique which aims to avoid over fitting. It splits the data into a series of training and testing splits, optimizes our chosen parameters on the training data, and sees how well the strategy performs on the testing data.

Download stock price data for a single ticker for walk-forward analysis

In [ ]:
start = "2015"
end = "now"
prices = vbt.YFData.download("AAPL", start=start, end=end).get("Close")

In [ ]:
prices

Define moving average window combinations for testing

In [ ]:
windows = np.arange(10, 51)

In [ ]:
windows

Perform rolling split to create in-sample and out-of-sample datasets for walk-forward analysis

In [ ]:
(in_price, in_indexes), (out_price, out_indexes) = prices.vbt.rolling_split(
    n=30,
    window_len=365 * 2,
    set_lens=(180,),
    left_to_right=False,
    trace_names=["train", "test"],
)

In [ ]:
print(in_price.shape, len(in_indexes))
print(out_price.shape, len(out_indexes))

Visualize the rolling split for walk-forward analysis

In [ ]:
prices.vbt.rolling_split(
    n=30,
    window_len=365 * 2,
    set_lens=(180,),
    left_to_right=False,
    trace_names=["train", "test"],
    plot=True,
)

Define a helper function to simulate all parameter combinations and return the Sharpe ratio

In [ ]:
def simulate_all_params(price, windows, **kwargs):
    fast_ma, slow_ma = vbt.MA.run_combs(
        price, windows, r=2, short_names=["fast", "slow"]
    )

    entries = fast_ma.ma_crossed_above(slow_ma)
    exits = fast_ma.ma_crossed_below(slow_ma)

    pf = vbt.Portfolio.from_signals(price, entries, exits, **kwargs)

    return pf.sharpe_ratio()

Define a helper function to get the best parameters given their indexes

In [ ]:
def get_best_params(performance, level_name):
    idx = performance[performance.groupby("split_idx").idxmax()].index
    return idx.get_level_values(level_name).to_numpy()

Define a helper function to get the indexes of the best parameter combinations

In [ ]:
def get_best_index(performance):
    return performance[performance.groupby("split_idx").idxmax()].index

Define a helper function to simulate the best parameters for each split

In [ ]:
def simulate_best_params(price, best_fast_windows, best_slow_windows, **kwargs):
    fast_ma = vbt.MA.run(price, window=best_fast_windows, per_column=True)
    slow_ma = vbt.MA.run(price, window=best_slow_windows, per_column=True)

    entries = fast_ma.ma_crossed_above(slow_ma)
    exits = fast_ma.ma_crossed_below(slow_ma)

    pf = vbt.Portfolio.from_signals(price, entries, exits, **kwargs)
    return pf.sharpe_ratio()

Simulate all moving average window combinations for in-sample data to find the best Sharpe ratio

In [ ]:
in_sharpe = simulate_all_params(in_price, windows, direction="both", freq="d")
in_sharpe

Get the indexes of the best parameter combinations for in-sample data

In [ ]:
in_best_index = get_best_index(in_sharpe)
in_best_index

Get the fast moving average windows that maximize the in-sample Sharpe ratio for each split

In [ ]:
in_best_fast_windows = get_best_params(in_sharpe, "fast_window")
in_best_fast_windows

Get the slow moving average windows that maximize the in-sample Sharpe ratio for each split

In [ ]:
in_best_slow_windows = get_best_params(in_sharpe, "slow_window")
in_best_slow_windows

Combine the best moving average window pairs

In [ ]:
in_best_window_pairs = np.array(list(zip(in_best_fast_windows, in_best_slow_windows)))
in_best_window_pairs

Simulate all moving average window combinations for out-of-sample data to find the best Sharpe ratio

In [ ]:
out_sharpe = simulate_all_params(out_price, windows, direction="both", freq="d")
out_sharpe

Evaluate the performance of the best in-sample parameters on out-of-sample data

In [ ]:
out_test_sharpe = simulate_best_params(
    out_price, in_best_fast_windows, in_best_slow_windows, direction="both", freq="d"
)
out_test_sharpe

Calculate the median Sharpe ratio for in-sample data and the test Sharpe ratio for out-of-sample data

In [ ]:
in_sample_median = in_sharpe.groupby("split_idx").median().values
out_sample_test = out_test_sharpe.values
len(in_sample_median), len(out_sample_test)

Run a one-sided t-test to compare the mean Sharpe ratios of in-sample and out-of-sample data

In [ ]:
t, p = stats.ttest_ind(
    a=out_sample_test, 
    b=in_sample_median,
    # ‘greater’: the mean of the distribution underlying the first sample 
    # is greater than the mean of the distribution underlying the second sample.
    alternative="greater"
)
t, p

<h3><a name="your-turn">Your Turn</a></h3>(<a href="#0">Go to top</a>)<p>Well done on completing the module! Now, it's time for a brief knowledge assessment.</p><div style="border: 4px solid coral; text-align: center; margin: auto;"><h2><i>Try it Yourself!</i></h2><p style="text-align: center; margin: auto;">As a general statement, simple moving average strategies won't consistently make money. However, it's important to fully investigate a strategy before dismissing it. Our example only tests periods 10 through 50. Experiment with a different range of windows (e.g. 50 to 200). Does the p-value improve? How about a longer date range and more splits? </div>